In [2]:
import mujoco
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- MODEL DEFINITION ---
xml = """
<mujoco>
    <option gravity="0 0 -9.81"/>
    <worldbody>
        <camera name="sideview" pos="4 0 0.5" euler="0 90 90"/>
        <geom name="ground" type="plane" size="2 2 0.1" pos="0 0 0" rgba="0 0.6 0 1" friction="1 0.005 0.0001"/>
        <body name="rodd" pos="0 -0.5 1">
            <freejoint/>
            <geom name="rod" type="cylinder" pos="0 0 0" size="0.05 0.25" rgba="0.2 0.2 0.8 1" solref="0.0001 0.001" solimp="0.99 0.99 0.01" friction="1 0.005 0.0001"/>
        </body>
    </worldbody>
</mujoco>
"""

def run_simulation(parameters):

    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model)

    # --- INITIAL STATE SETUP ---
    data.qpos[0] = 0      # x
    data.qpos[1] = -1.5   # y
    data.qpos[2] = 1      # z
    angle = np.deg2rad(60)
    data.qpos[3] = np.cos(angle / 2)  # w
    data.qpos[4] = np.sin(angle / 2)  # x
    data.qpos[5] = 0                  # y
    data.qpos[6] = 0                  # z

    data.qvel[0] = 0.0   # vx
    data.qvel[1] = 5.0   # vy
    data.qvel[2] = -5.0  # vz
    data.qvel[3] = -5.0  # wx
    data.qvel[4] = 0.0   # wy
    data.qvel[5] = 0.0   # wz

    # --- SIMULATION LOOP AND DATA COLLECTION ---
    positions = []
    linear_velocities = []
    angular_velocities = []
    external_forces = []
    saved_qpos = []
    saved_qvel = []

    num_steps = 600
    # Get the geom ID for the rod (before the loop)
    rod_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "rod")
    for _ in range(num_steps):
        mujoco.mj_step(model, data)
        positions.append(data.xipos[1].copy())
        linear_velocities.append(data.qvel[:3].copy())
        angular_velocities.append(data.qvel[3:].copy())
        saved_qpos.append(data.qpos.copy())
        saved_qvel.append(data.qvel.copy())


        # Inside the simulation loop:
        net_force = np.zeros(3)
        for i in range(data.ncon):
            contact = data.contact[i]
            if contact.geom1 == rod_geom_id or contact.geom2 == rod_geom_id:
                force = np.zeros(6)
                mujoco.mj_contactForce(model, data, i, force)
                net_force += force[:3]
        external_forces.append(net_force.copy())

    # ...existing code...

    # Track number of contacts during the simulation
    num_contacts = []
    for _ in range(num_steps):
        mujoco.mj_step(model, data)
        positions.append(data.xipos[1].copy())
        linear_velocities.append(data.qvel[:3].copy())
        angular_velocities.append(data.qvel[3:].copy())
        saved_qpos.append(data.qpos.copy())
        saved_qvel.append(data.qvel.copy())
        num_contacts.append(data.ncon)
        # ...rest of your loop...

    # After the simulation loop, find the first collision duration
    first_contact = None
    first_contact_end = None
    for i, n in enumerate(num_contacts):
        if n > 0 and first_contact is None:
            first_contact = i
        if first_contact is not None and n == 0:
            first_contact_end = i
            break
    if first_contact is not None and first_contact_end is not None:
        print(f"First collision lasts {first_contact_end - first_contact} steps")
    elif first_contact is not None:
        print(f"First collision starts at step {first_contact} and lasts until the end of the simulation")
    else:
        print("No collision detected in the simulation")

    # Convert to NumPy arrays
    positions = np.array(positions)
    linear_velocities = np.array(linear_velocities)
    angular_velocities = np.array(angular_velocities)
    external_forces = np.array(external_forces)
    saved_qpos = np.array(saved_qpos)
    saved_qvel = np.array(saved_qvel)

    return positions, linear_velocities, angular_velocities, external_forces, saved_qpos, saved_qvel, model, data, renderer

friction = np.linspace(0.3,0.7,5)

# --- PLOTTING ---

# 1. Position
plt.figure()
plt.plot(positions[:, 0], label='z')
plt.plot(positions[:, 1], label='x')
plt.plot(positions[:, 2], label='y')
plt.xlabel('Time step')
plt.ylabel('Position (m)')
plt.title('Cylinder Position Throughout Simulation')
plt.legend()
plt.grid(True)
plt.show()

# 2. Linear Velocity
plt.figure()
plt.plot(linear_velocities[:, 0], label='vz')
plt.plot(linear_velocities[:, 1], label='vx')
plt.plot(linear_velocities[:, 2], label='vy')
plt.xlabel('Time step')
plt.ylabel('Linear Velocity (m/s)')
plt.title('Cylinder Linear Velocity Over Time')
plt.legend()
plt.grid(True)
plt.show()

# 3. Angular Velocity
plt.figure()
plt.plot(angular_velocities[:, 0], label='wz')
plt.plot(angular_velocities[:, 1], label='wx')
plt.plot(angular_velocities[:, 2], label='wy')
plt.xlabel('Time step')
plt.ylabel('Angular Velocity (rad/s)')
plt.title('Cylinder Angular Velocity Over Time')
plt.legend()
plt.grid(True)
plt.show()

# 4. External Forces
plt.figure()
plt.plot(external_forces[:, 0], label='Fz')
plt.plot(external_forces[:, 1], label='Fx')
plt.plot(external_forces[:, 2], label='Fz')
plt.xlabel('Time step')
plt.ylabel('External Force (N)')
plt.title('External Forces on Rod Over Time')
plt.legend()
plt.grid(True)
plt.show()

# --- ANIMATION USING SAVED STATES ---
frames = []
num_frames = 100  # or any number <= num_steps

# Create a new data object for rendering (optional, for safety)
data_render = mujoco.MjData(model)

for i in range(num_frames):
    data_render.qpos[:] = saved_qpos[i]
    data_render.qvel[:] = saved_qvel[i]
    mujoco.mj_forward(model, data_render)  # update derived quantities
    renderer.update_scene(data_render, camera="sideview")
    frame = renderer.render()
    frames.append(frame)

fig, ax = plt.subplots()
im = ax.imshow(frames[0])
ax.axis('off')

def update(i):
    im.set_data(frames[i])
    return [im]

ani = FuncAnimation(fig, update, frames=num_frames, interval=20, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

NameError: name 'positions' is not defined

<Figure size 640x480 with 0 Axes>

dependent variables:
max height,
incident angle,
contact time,
orientation at max height,
impulse,

independent variables:
rod length,
x, y, w velocity
friction (relationship with optimal angle)

